# Spam Classifier Walkthrough

Step-by-step notebook for the spam/ham email classifier on the SpamAssassin corpus.

The canonical implementation lives in the [`classifier/`](classifier/) package (`io`, `preprocess`, `features`, `model`, `viz`). [`spam_classifier.py`](spam_classifier.py) is a thin entry point and re-exports the public API. This notebook imports those functions phase by phase so you can inspect intermediate results.

**Run from the repo root** so `data/spamassassin/` resolves correctly.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from sklearn.naive_bayes import ComplementNB

from spam_classifier import (
    DATA_DIR,
    FIGURES_DIR,
    load_emails,
    explore_dataset,
    plot_class_and_content_distribution,
    preprocess_all_emails,
    display_preprocessing_examples,
    remove_empty_emails,
    create_bow_features,
    analyze_bow_features,
    determine_optimal_clusters,
    fit_clustering_model,
    analyze_clusters,
    add_cluster_features,
    train_with_cross_validation,
    create_confusion_matrix_plot,
    generate_classification_metrics,
    analyze_misclassifications,
    threshold_analysis,
    create_additional_visualizations,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data: {DATA_DIR.resolve()}")
print(f"Figures: {FIGURES_DIR.resolve()}")

## Phase 1: Data loading and exploration

Load emails from the five SpamAssassin directories (easy_ham, easy_ham_2, hard_ham, spam, spam_2). Each file is parsed with Latin-1 encoding; labels are assigned by folder (ham = 0, spam = 1).

In [ ]:
emails, labels, raw_texts = load_emails()
explore_dataset(emails, labels)

class_content_fig = plot_class_and_content_distribution(emails, labels)
class_content_fig.savefig(FIGURES_DIR / "00_class_and_content_distribution.png", dpi=150, bbox_inches="tight")
plt.close(class_content_fig)

## Phase 2: Text preprocessing

Extract body text (and subject), strip HTML, normalize tokens, and remove stopwords. Empty emails are dropped before feature engineering.

In [ ]:
cleaned_emails, empty_indices = preprocess_all_emails(emails, include_subject=True)
display_preprocessing_examples(emails, labels, cleaned_emails, num_examples=2)

if empty_indices:
    emails, cleaned_emails, labels, raw_texts = remove_empty_emails(
        emails, cleaned_emails, labels, raw_texts, empty_indices
    )

## Phase 3: TF-IDF Bag-of-Words features

TF-IDF with 3,000 features, unigrams and bigrams. Term weighting emphasizes distinctive spam/ham vocabulary.

In [ ]:
bow_matrix, vectorizer = create_bow_features(
    cleaned_emails,
    max_features=3000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
)

feature_analysis_fig = analyze_bow_features(bow_matrix, vectorizer, labels, top_n=20)
feature_analysis_fig.savefig(FIGURES_DIR / "01_bow_feature_analysis.png", dpi=150, bbox_inches="tight")
plt.close(feature_analysis_fig)

## Phase 4: Clustering features

Mini-Batch K-Means on the TF-IDF matrix. Cluster count is chosen by silhouette score ($k = 30$). One-hot cluster assignments are concatenated with the BOW matrix.

In [ ]:
cluster_metrics, cluster_eval_fig = determine_optimal_clusters(
    bow_matrix, k_range=range(2, 31), sample_size=3000
)
cluster_eval_fig.savefig(FIGURES_DIR / "02_cluster_evaluation.png", dpi=150, bbox_inches="tight")
plt.close(cluster_eval_fig)

optimal_k = cluster_metrics["best_silhouette_k"]
print(f"Selected k = {optimal_k}")

In [ ]:
clustering_model, cluster_labels = fit_clustering_model(bow_matrix, n_clusters=optimal_k)

cluster_info, cluster_analysis_fig = analyze_clusters(
    cluster_labels, labels, cleaned_emails, vectorizer, bow_matrix
)
cluster_analysis_fig.savefig(FIGURES_DIR / "03_cluster_analysis.png", dpi=150, bbox_inches="tight")
plt.close(cluster_analysis_fig)

combined_matrix = add_cluster_features(bow_matrix, cluster_labels)
print(f"Combined feature matrix shape: {combined_matrix.shape}")

## Phase 5: Complement Naive Bayes with cross-validation

ComplementNB handles class imbalance. Stratified 10-fold CV produces out-of-fold predictions for every sample (no train/test split, no data leakage).

In [ ]:
classifier = ComplementNB(alpha=1.0)
print("Classifier: ComplementNB (alpha=1.0)")

y_pred, y_pred_proba, fold_scores, cv_object = train_with_cross_validation(
    X=combined_matrix,
    y=labels,
    classifier=classifier,
    n_folds=10,
)

## Phase 6: Evaluation and visualizations

Confusion matrix, classification metrics, misclassification analysis, threshold/ROC analysis, and summary plots — all on out-of-fold predictions.

In [ ]:
confusion_fig, cm = create_confusion_matrix_plot(labels, y_pred)
confusion_fig.savefig(FIGURES_DIR / "04_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close(confusion_fig)

metrics_dict = generate_classification_metrics(labels, y_pred)

In [ ]:
fp_indices, fn_indices = analyze_misclassifications(
    labels, y_pred, cleaned_emails, labels, num_examples=10
)

In [ ]:
threshold_fig, threshold_metrics = threshold_analysis(labels, y_pred_proba)
threshold_fig.savefig(FIGURES_DIR / "05_threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.close(threshold_fig)

summary_fig = create_additional_visualizations(fold_scores, metrics_dict)
summary_fig.savefig(FIGURES_DIR / "06_performance_summary.png", dpi=150, bbox_inches="tight")
plt.close(summary_fig)

## Final summary

In [ ]:
import numpy as np

print(f"Final Accuracy (all folds): {metrics_dict['accuracy']:.4f}")
print(f"Mean CV Accuracy: {np.mean(fold_scores):.4f} (+/-{np.std(fold_scores):.4f})")
print(f"\nHam  — Precision: {metrics_dict['ham_precision']:.4f}, Recall: {metrics_dict['ham_recall']:.4f}")
print(f"Spam — Precision: {metrics_dict['spam_precision']:.4f}, Recall: {metrics_dict['spam_recall']:.4f}")
print(f"\nROC AUC: {threshold_metrics['roc_auc']:.4f}")
print(f"\nFigures saved to: {FIGURES_DIR.resolve()}")